In [ ]:
import os
import boto3

# Initialize S3 client and resource for Allas
s3_client = boto3.client("s3", endpoint_url='https://a3s.fi')
s3_resource = boto3.resource("s3", endpoint_url='https://a3s.fi')

bucket_name = "MAPICO_FIN_CITIES_EMISSION_2024_CAR"
prefix = "Oulu_region_30MAY24_wCO2_CAR/"  

destination_dir = "scratch/Oulu_region_30MAY24_wCO2_CAR"
os.makedirs(destination_dir, exist_ok=True)

bucket = s3_resource.Bucket(bucket_name)

downloaded = 0
skipped = 0

for obj in bucket.objects.filter(Prefix=prefix):
    key = obj.key

    # Remove the prefix so the full remote path is not recreated locally
    relative_path = os.path.relpath(key, prefix)
    local_file_path = os.path.join(destination_dir, relative_path)

    # Create subdirectories if needed
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    if os.path.exists(local_file_path):
        print(f"Skipping {key}, already exists locally.")
        skipped += 1
        continue

    try:
        print(f"Downloading {key} → {local_file_path}")
        bucket.download_file(key, local_file_path)
        downloaded += 1
    except Exception as e:
        print(f"Failed to download {key}. Error: {e}")

print(f"\nDownloaded: {downloaded}, Skipped: {skipped}")

In [ ]:
import pandas as pd
import os
import glob

# Path to directory with parquet files
dir_path = "scratch/Oulu_region_30MAY24_wCO2_CAR"

# Find all .parquet files
parquet_files = glob.glob(os.path.join(dir_path, "**/*.parquet"), recursive=True)

# Check if any files were found
if not parquet_files:
    raise FileNotFoundError("No .parquet files found in the specified directory.")

# Read and concatenate all parquet files into a single DataFrame
df_list = [pd.read_parquet(fp) for fp in parquet_files]
df_all = pd.concat(df_list, ignore_index=True)

# Display basic info
print(f"Loaded {len(parquet_files)} parquet files.")
print(df_all.head())

In [ ]:
df_all.shape

In [ ]:
df_all.head()

In [ ]:
# Filter rows where pt_time is less than or equal to 15, 30, and 45
df_car_co2_250 = df_all[df_all["car_co2"] <= 250].copy()
df_car_co2_500 = df_all[df_all["car_co2"] <= 500].copy()

In [ ]:
df_car_co2_2000 = df_all[df_all["car_co2"] <= 2000].copy()

In [ ]:
df_car_co2_6000 = df_all[df_all["car_co2"] <= 6000].copy()

In [ ]:
df_car_co2_6000.sort_values("parking_driving_dist").head(1000)


In [ ]:
df_car_co2_6000.to_parquet("scratch/car_co2_6000_oulu.parquet")

In [ ]:
df_car_co2_6000

In [ ]:
hsk_pois = pd.read_parquet("./data/pois_per_hex.parquet")

In [ ]:
hsk_pois

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = (
    hsk_pois
    .groupby(['h3_id', 'category'])['count']
    .sum()                           # sum the values instead of counting rows
    .unstack(fill_value=0)           # make wide format, categories as columns
    .reset_index()
)

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_car_co2_2000_merged = df_car_co2_2000.merge(pois_grouped, how='left', left_on='Destination_Hexagon_ID', right_on='h3_id')
df_car_co2_2000_merged = df_car_co2_2000_merged.drop(columns=['h3_id'])



# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
   df_car_co2_2000_merged[col] = df_car_co2_2000_merged[col].fillna(0) + 0

In [ ]:
df_car_co2_2000_merged.to_parquet("./output/cars_co2_all_4000.parquet")

In [ ]:
# List the category columns
category_cols = [
    "Educational Facilities",
    "Grocery Stores & Supermarkets",
    "Jobs, Professional Services & Religious",
    "Restaurant & Entertainment",
    "Shopping & Retail",
    "Uncategorized",
    "Well-being & Lifestyle"
]

# Add total opportunities column
df_car_co2_2000_merged["total_opportunities"] = df_car_co2_2000_merged[category_cols].sum(axis=1)


In [ ]:
# Mapping from hex_id to place name
hex_ids = [
    "891126d3313ffff", "890899683d3ffff", "891126d191bffff",
    "891126d05afffff", "891126d707bffff", "891126d04c3ffff",
    "8908996d193ffff"
]

place_names = ["City Center", "Suurpelto", "Itäkeskus", "Oulunkylä", "Vanta Center", "Pasila", "Kehä 3"]

hex_to_name = dict(zip(hex_ids, place_names))

# Add a column with names
df_subset = df_car_co2_2000_merged[df_car_co2_2000_merged["Origin_Hexagon_ID"].isin(hex_ids)].copy()
df_subset["Origin_Name"] = df_subset["Origin_Hexagon_ID"].map(hex_to_name)

# Build cumulative curves per origin name
df_curves = (
    df_subset
    .sort_values(["Origin_Name", "car_co2"])
    .groupby("Origin_Name", group_keys=False)
    .apply(lambda g: g.assign(
        cumulative_opportunities=g["total_opportunities"].cumsum()
    ))
)

import matplotlib.pyplot as plt

plt.figure(figsize=(10,7))

for origin, g in df_curves.groupby("Origin_Name"):
    plt.plot(g["car_co2"], g["cumulative_opportunities"], marker="o", label=origin)

plt.xlabel("Car CO₂ emissions")
plt.ylabel("Cumulative opportunities reached")
plt.title("Accessibility vs. Car CO₂ Emissions per Origin")
plt.legend(title="Origin", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
df_curve

In [ ]:
# Step 1: Group hsk_pois by 'h3_id' and 'category' to get counts
pois_grouped = hsk_pois.groupby(['h3_id', 'category']).size().unstack(fill_value=0).reset_index()

# Step 2: Merge with df_pt_15 using 'to_id' (in df_pt_15) and 'h3_id' (in pois_grouped)
df_car_co2_500_merged = df_car_co2_500.merge(pois_grouped, how='left', left_on='Destination_Hexagon_ID', right_on='h3_id')
df_car_co2_500_merged = df_car_co2_500_merged.drop(columns=['h3_id'])

# Step 3: Merge again using 'from_id' to get POIs at origin
from_pois = df_car_co2_500.merge(pois_grouped, how='left', left_on='Origin_Hexagon_ID', right_on='h3_id')
from_pois = from_pois.drop(columns=['h3_id'])

# Step 4: Add category columns from origin and destination
category_cols = pois_grouped.columns.drop('h3_id')
for col in category_cols:
   df_car_co2_500_merged[col] = df_car_co2_500_merged[col].fillna(0) + from_pois[col].fillna(0)

In [ ]:
# Step 5: Group by 'from_id', sum POI categories, and average pt_co2
category_cols = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Uncategorized',
    'Well-being & Lifestyle'
]

grouped_summary = df_car_co2_500_merged.groupby('Origin_Hexagon_ID')[category_cols + ['car_time']].agg({
    'Educational Facilities': 'sum',
    'Grocery Stores & Supermarkets': 'sum',
    'Jobs, Professional Services & Religious': 'sum',
    'Restaurant & Entertainment': 'sum',
    'Shopping & Retail': 'sum',
    'Uncategorized': 'sum',
    'Well-being & Lifestyle': 'sum',
    'car_time':'mean'
    
}).reset_index()

In [ ]:
grouped_summary

In [ ]:
# Step 6: Create total_pois column (excluding 'Uncategorized')
grouped_summary['total_pois'] = grouped_summary[[
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]].sum(axis=1)

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

# Function to convert h3 to polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Convert 'from_id' to geometry
grouped_summary = grouped_summary.copy()
geometry = grouped_summary['Origin_Hexagon_ID'].apply(h3_to_polygon)
gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')

# Classify total_pois into 5 natural breaks
classifier = mapclassify.NaturalBreaks(y=gdf_hexes['total_pois'], k=5)
gdf_hexes['poi_class'] = classifier.yb

# Get bin edges for legend
bin_edges = classifier.bins

# Define custom legend handles
legend_handles = []
for i in range(len(bin_edges)):
    if i == 0:
        label = f"<= {bin_edges[i]:.1f}"
    else:
        label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
    patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges)-1)), label=label)
    legend_handles.append(patch)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
ax.set_title("Total POIs per Hexagon (Natural Breaks)")
ax.axis('off')
plt.legend(handles=legend_handles, title="Total POIs", loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon
import contextily as ctx; import basemaps
import matplotlib.pyplot as plt
import mapclassify
import matplotlib.patches as mpatches

# Function to convert h3 index to shapely Polygon
def h3_to_polygon(h):
    return Polygon(h3.h3_to_geo_boundary(h, geo_json=True))

# Define POI categories to plot
poi_categories = [
    'Educational Facilities',
    'Grocery Stores & Supermarkets',
    'Jobs, Professional Services & Religious',
    'Restaurant & Entertainment',
    'Shopping & Retail',
    'Well-being & Lifestyle'
]

# Create GeoDataFrame with geometry if not already created
if 'geometry' not in grouped_summary.columns:
    geometry = grouped_summary['Origin_Hexagon_ID'].apply(h3_to_polygon)
    gdf_hexes = gpd.GeoDataFrame(grouped_summary, geometry=geometry, crs='EPSG:4326')
else:
    gdf_hexes = grouped_summary.copy()

# Set up plot grid
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

# Generate map for each category
for idx, category in enumerate(poi_categories):
    ax = axes[idx]
    # Classify with Natural Breaks
    classifier = mapclassify.NaturalBreaks(y=gdf_hexes[category], k=5)
    gdf_hexes['poi_class'] = classifier.yb
    bin_edges = classifier.bins

    # Create custom legend
    legend_handles = []
    for i in range(len(bin_edges)):
        if i == 0:
            label = f"<= {bin_edges[i]:.1f}"
        else:
            label = f"> {bin_edges[i-1]:.1f} – {bin_edges[i]:.1f}"
        patch = mpatches.Patch(color=plt.cm.viridis(i / (len(bin_edges) - 1)), label=label)
        legend_handles.append(patch)

    # Plot
    gdf_hexes.plot(ax=ax, column='poi_class', cmap='viridis', legend=False, edgecolor='black', linewidth=0.2)
    ctx.add_basemap(ax, source=basemaps.POSITRON, crs=gdf_hexes.crs.to_string())
    ax.set_title(category)
    ax.axis('off')
    ax.legend(handles=legend_handles, title=category, loc='upper left')

plt.suptitle("POI Categories by Hexagon in a 500g trip by car (Natural Breaks)", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()
